In [110]:
import modin.pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

In [111]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
import warnings
warnings.filterwarnings('ignore')

#### Dataset link https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=links.csv
#### **movies_metadata.csv**: The main Movies Metadata file. Contains information on 45,000 movies featured in the Full MovieLens dataset. Features include posters, backdrops, budget, revenue, release dates, languages, production countries and companies.

#### **keywords.csv**: Contains the movie plot keywords for our MovieLens movies. Available in the form of a stringified JSON Object.

#### **credits.csv**: Consists of Cast and Crew Information for all our movies. Available in the form of a stringified JSON Object.

#### **links.csv**: The file that contains the TMDB and IMDB IDs of all the movies featured in the Full MovieLens dataset.

#### **links_small.csv**: Contains the TMDB and IMDB IDs of a small subset of 9,000 movies of the Full Dataset.

#### **ratings_small.csv**: The subset of 100,000 ratings from 700 users on 9,000 movies.

#### The Full MovieLens Dataset consisting of 26 million ratings and 750,000 tag applications from 270,000 users on all the 45,000 movies in this dataset can be accessed here



In [112]:
# Load datasets
credits_df = pd.read_csv('dataset/credits.csv')
keywords_df = pd.read_csv('dataset/keywords.csv')
links_small_df = pd.read_csv('dataset/links_small.csv')
links_df = pd.read_csv('dataset/links.csv')
movies_metadata_df = pd.read_csv('dataset/movies_metadata.csv', low_memory=False)
ratings_small_df = pd.read_csv('dataset/ratings_small.csv')
ratings_df = pd.read_csv('dataset/ratings.csv')

In [113]:
keywords_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB


In [114]:
links_small_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9125 non-null   int64  
 1   imdbId   9125 non-null   int64  
 2   tmdbId   9112 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 214.0 KB


In [115]:
links_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 45843 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  45843 non-null  int64  
 1   imdbId   45843 non-null  int64  
 2   tmdbId   45624 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 1.0 MB


In [116]:
movies_metadata_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 1

In [117]:
movies_metadata_df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [118]:
# Filtering records where movies title is null
movies_metadata_df = movies_metadata_df[~movies_metadata_df['title'].isnull()]
movies_metadata_df = movies_metadata_df[~movies_metadata_df['overview'].isnull()]

In [119]:
movies_metadata_df.isnull().sum()

adult                        0
belongs_to_collection    40075
budget                       0
genres                       0
homepage                 36745
id                           0
imdb_id                     15
original_language           10
original_title               0
overview                     0
popularity                   0
poster_path                343
production_companies         0
production_countries         0
release_date                71
revenue                      0
runtime                      0
spoken_languages             0
status                      65
tagline                  24102
title                        0
video                        0
vote_average                 0
vote_count                   0
dtype: int64

In [120]:
ratings_small_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [121]:
ratings_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 26024289 entries, 0 to 26024288
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 794.2 MB


In [122]:
# Explore datasets
credits_df.info()

#Id, cast

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


In [123]:
credits_df.head()

cast  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         [{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg'}, {'cast_id': 19, 'character': 'Hamm (voice)', 'credit_id': '52fe4284c3a36847f8024fa9', 'gender': 2, 'id': 7907, 'name': 'John Ratzenberger', 'order': 5, 'profile_path': '/yGechiKWL6TJDfVE2KPSJYqdMsY.jpg'}, {'cast_id': 20, 'character': 'Bo Peep (voice)', 'credit_id': '52fe4284c3a36847f8024fad', 'gender': 1, 'id': 8873, 'name': 'Annie Potts', 'order': 6, 'profile_path': '/eryXT84RL41jHSJcMy4kS3u9y6w.jpg'}, {'cast_id': 26, 'character': 'Andy (voice)', 'credit_id': '52fe4284c3a36847f8024fc1', 'gender': 0, 'id': 1116442, 'name': 'John Morris', 'order': 7, 'profile_path': '/vYG

In [124]:
movies_metadata_df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",21.946943,1995-10-30,373554033,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.",17.015539,1995-12-15,262797249,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqgPgm3F85NMCii9gVFUcet.jpg', 'backdrop_path': '/hypTnLot2z8wpFS7qwsQHW1uV8u.jpg'}",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",NaN,15602,tt0113228,en,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming the locals who worry she'll scare the fish away. But she's less interested in seafood than she is in cooking up a hot time with Max.",11.712900,1995-12-22,0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to break a string of less-than-stellar lovers. Friends and confidants Vannah, Bernie, Glo and Robin talk it all out, determined to find a better way to breathe.",3.859495,1995-12-22,81452156,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself... and never let you forget it.,Waiting to Exhale,False,6.1,34
4,False,"{'id': 96871, 'name': 'Father of the Bride Collection', 'poster_path': '/nts4iOmNnq7GNicycMJ9pSAn204.jpg', 'backdrop_path': '/7qwE57OVZmMJChBpLEbJEmzUydk.jpg'}",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,"Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and that George's wife, Nina, is expecting too. He was planning on selling their home, but that's a plan that -- like George -- will have to change with the arrival of both a grandchild and a kid of his own.",8.387519,1995-02-10,76578911,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's In For The Surprise Of His Life!,Father of the Bride Part II,False,5.7,173


In [125]:
keywords_df.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]"
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 10941, 'name': 'disappearance'}, {'id': 15101, 'name': ""based on children's book""}, {'id': 33467, 'name': 'new home'}, {'id': 158086, 'name': 'recluse'}, {'id': 158091, 'name': 'giant insect'}]"
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392, 'name': 'best friend'}, {'id': 179431, 'name': 'duringcreditsstinger'}, {'id': 208510, 'name': 'old men'}]"
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id': 10131, 'name': 'interracial relationship'}, {'id': 14768, 'name': 'single mother'}, {'id': 15160, 'name': 'divorce'}, {'id': 33455, 'name': 'chick flick'}]"
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'name': 'midlife crisis'}, {'id': 2246, 'name': 'confidence'}, {'id': 4995, 'name': 'aging'}, {'id': 5600, 'name': 'daughter'}, {'id': 10707, 'name': 'mother daughter relationship'}, {'id': 13149, 'name': 'pregnancy'}, {'id': 33358, 'name': 'contraception'}, {'id': 170521, 'name': 'gynecologist'}]"


In [126]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [127]:
ratings_small_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [128]:
links_small_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [129]:
links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


#### Columns selected for the algorithm training are 'id', 'genres', 'original_language', 'overview', 'title', 'keywords', 'cast', 'rating', 'movieId'

#### Few columns will be transformed and few will be dropped later once transformation is done and later not required

In [130]:
import ast
import json
import pandas as pd
from itertools import chain

def parse_cast(cell, name_keys):
    """Robust parser for the `cast` column. Returns a list of actor names (strings).
    name_keys may be a single key (str) or an iterable of keys to try in order.
    Supports: JSON strings, Python-literal strings (single quotes), already-parsed lists/dicts, and empty / NaN.
    """
    # normalize name_keys to an ordered iterable of strings
    if isinstance(name_keys, str):
        name_keys = (name_keys,)

    if pd.isna(cell) or cell == '':
        return []
    # If it's already a list, assume it contains dicts or strings
    if isinstance(cell, list):
        parsed = cell
    else:
        # try a few common parsers
        parsed = None
        for parser in (ast.literal_eval, json.loads):
            try:
                parsed = parser(cell)
                break
            except (ValueError, SyntaxError, json.JSONDecodeError):
                continue
        # last-ditch: try replacing single quotes then json.loads
        if parsed is None:
            try:
                parsed = json.loads(cell.replace("'", '\"'))
            except Exception:
                # can't parse
                return []

    names = []
    if isinstance(parsed, list):
        for item in parsed:
            if isinstance(item, dict):
                # try each candidate key in order
                found = None
                for k in name_keys:
                    found = item.get(k)
                    if found:
                        break
                if found:
                    names.append(str(found).strip())
            elif isinstance(item, str):
                names.append(item.strip())
    elif isinstance(parsed, dict):
        # maybe a single object
        found = None
        for k in name_keys:
            found = parsed.get(k)
            if found:
                break
        if found:
            names.append(str(found).strip())
    # deduplicate while preserving order
    seen = set()
    uniq = []
    for n in names:
        if n and n not in seen:
            uniq.append(n)
            seen.add(n)
    return uniq


In [131]:
import ast
import json
import pandas as pd
def extract_director(cell):
    """Extract director name(s) from a `crew` cell."""
    if pd.isna(cell) or cell == '':
        return []
    # If already parsed as list/dict objects
    if isinstance(cell, list):
        parsed = cell
    else:
        # try JSON then Python literal parsing
        parsed = None
        try:
            parsed = json.loads(cell)
        except Exception:
            try:
                parsed = ast.literal_eval(cell)
            except Exception:
                return []
    directors = []
    for item in parsed:
        if not isinstance(item, dict):
            continue
        job = (item.get('job') or '').lower()
        if job == 'director':
            name = item.get('name')
            if name:
                directors.append(str(name).strip())
    # deduplicate while preserving order
    seen = set()
    out = []
    for d in directors:
        if d not in seen:
            out.append(d)
            seen.add(d)
    return out

In [132]:
# pass a single key name as a list via args=() to apply
credits_df['cast'] = credits_df['cast'].apply(parse_cast, args=(['name'],))
credits_df['director'] = credits_df['crew'].apply(extract_director)
keywords_df['keywords'] = keywords_df['keywords'].apply(parse_cast, args=(['name'],))
movies_metadata_df['genres'] = movies_metadata_df['genres'].apply(parse_cast, args=(['name'],))
movies_metadata_df['spoken_languages'] = movies_metadata_df['spoken_languages'].apply(parse_cast, args=(['name'],))

In [133]:
credits_df['director'].head()

0      [John Lasseter]
1       [Joe Johnston]
2      [Howard Deutch]
3    [Forest Whitaker]
4      [Charles Shyer]
Name: director, dtype: object

In [134]:
keywords_df['keywords'].head()

0                                [jealousy, toy, boy, friendship, friends, rivalry, boy next door, new toy, toy comes to life]
1                                       [board game, disappearance, based on children's book, new home, recluse, giant insect]
2                                                                        [fishing, best friend, duringcreditsstinger, old men]
3                                              [based on novel, interracial relationship, single mother, divorce, chick flick]
4    [baby, midlife crisis, confidence, aging, daughter, mother daughter relationship, pregnancy, contraception, gynecologist]
Name: keywords, dtype: object

In [135]:
movies_metadata_df['genres'].head()

0     [Animation, Comedy, Family]
1    [Adventure, Fantasy, Family]
2               [Romance, Comedy]
3        [Comedy, Drama, Romance]
4                        [Comedy]
Name: genres, dtype: object

In [136]:
movies_metadata_df['spoken_languages'].head()

0              [English]
1    [English, Français]
2              [English]
3              [English]
4              [English]
Name: spoken_languages, dtype: object

In [137]:
movies_metadata_df.info()

<class 'modin.pandas.dataframe.DataFrame'>
Index: 44506 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  44506 non-null  object 
 1   belongs_to_collection  4431 non-null   object 
 2   budget                 44506 non-null  object 
 3   genres                 44506 non-null  object 
 4   homepage               7761 non-null   object 
 5   id                     44506 non-null  object 
 6   imdb_id                44491 non-null  object 
 7   original_language      44496 non-null  object 
 8   original_title         44506 non-null  object 
 9   overview               44506 non-null  object 
 10  popularity             44506 non-null  object 
 11  poster_path            44163 non-null  object 
 12  production_companies   44506 non-null  object 
 13  production_countries   44506 non-null  object 
 14  release_date           44435 non-null  object 
 15  re

In [138]:
movies_metadata_df['overview']= movies_metadata_df['overview'].apply(lambda x:x.split())

In [139]:
# Merge credits and keywords on 'id'
df = credits_df[['id', 'cast']].merge(keywords_df[['id', 'keywords']], on='id')
# Merge with movies_metadata on 'id'
# Adding numerical features will pull results toward items similar in those numeric aspects and may introduce popularity bias. So we need to drop it for the 'content based recommender' system

df = df.merge(movies_metadata_df[['id', 'genres', 'overview', 'title']], on='id')


In [141]:
df[df['title']=='Music and Lyrics']['genres']

11608    [Comedy, Music, Romance]
Name: genres, dtype: object

In [142]:
df['context'] = df['cast'] + df['keywords'] + df['genres'] + df['overview'] 

In [143]:
df['context'] = df['context'].apply(lambda x: " ".join(x))

In [144]:
df.drop(columns=['id', 'cast', 'keywords', 'genres', 'overview'], inplace=True)

In [145]:
# ensure df has a stable index and title column
df = df.reset_index(drop=True)

In [146]:
df.head()

,title,context
0,Toy Story,"Tom Hanks Tim Allen Don Rickles Jim Varney Wallace Shawn John Ratzenberger Annie Potts John Morris Erik von Detten Laurie Metcalf R. Lee Ermey Sarah Freeman Penn Jillette jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life Animation Comedy Family Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."
1,Jumanji,"Robin Williams Jonathan Hyde Kirsten Dunst Bradley Pierce Bonnie Hunt Bebe Neuwirth David Alan Grier Patricia Clarkson Adam Hann-Byrd Laura Bell Bundy James Handy Gillian Barber Brandon Obray Cyrus Thiedeke Gary Joseph Thorup Leonard Zola Lloyd Berry Malcolm Stewart Annabel Kershaw Darryl Henriques Robyn Driscoll Peter Bryant Sarah Gilson Florica Vlad June Lion Brenda Lockmuller board game disappearance based on children's book new home recluse giant insect Adventure Fantasy Family When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures."
2,Grumpier Old Men,"Walter Matthau Jack Lemmon Ann-Margret Sophia Loren Daryl Hannah Burgess Meredith Kevin Pollak fishing best friend duringcreditsstinger old men Romance Comedy A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming the locals who worry she'll scare the fish away. But she's less interested in seafood than she is in cooking up a hot time with Max."
3,Waiting to Exhale,"Whitney Houston Angela Bassett Loretta Devine Lela Rochon Gregory Hines Dennis Haysbert Michael Beach Mykelti Williamson Lamont Johnson Wesley Snipes based on novel interracial relationship single mother divorce chick flick Comedy Drama Romance Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to break a string of less-than-stellar lovers. Friends and confidants Vannah, Bernie, Glo and Robin talk it all out, determined to find a better way to breathe."
4,Father of the Bride Part II,"Steve Martin Diane Keaton Martin Short Kimberly Williams-Paisley George Newbern Kieran Culkin BD Wong Peter Michael Goetz Kate McGregor-Stewart Jane Adams Eugene Levy Lori Alan baby midlife crisis confidence aging daughter mother daughter relationship pregnancy contraception gynecologist Comedy Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and that George's wife, Nina, is expecting too. He was planning on selling their home, but that's a plan that -- like George -- will have to change with the arrival of both a grandchild and a kid of his own."


In [147]:
# Instead of creating a column for every single word in your dataset (which could be millions), it only picks the top 5,000 most frequent words. This keeps your data manageable.

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
tf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tf.fit_transform(df['context'])  # sparse matrix, shape (n_movies, n_features)

In [148]:
nn = NearestNeighbors(n_neighbors=6, metric='cosine', n_jobs=-1)
nn.fit(X)

,n_neighbors,6
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [149]:
def recommend(title, n=4):
    if title not in df['title'].values:
        raise KeyError(f"Title not in dataset: {title}")
    idx = int(df.index[df['title'] == title][0])
    dists, inds = nn.kneighbors(X[idx], n_neighbors=n+1)
    inds = inds.flatten()[1:]     # drop self
    return df['title'].iloc[inds].tolist()

In [151]:
def recommend(title, n=4):
    if title not in df['title'].values:
        raise KeyError(f"Title not in dataset: {title}")
    idx = int(df.index[df['title'] == title][0])
    dists, inds = nn.kneighbors(X[idx], n_neighbors=n+1)
    dists = dists.flatten()[1:]   # drop self
    inds = inds.flatten()[1:]
    sims = 1 - dists              # cosine similarity = 1 - distance
    results = list(zip(df['title'].iloc[inds].tolist(), sims.tolist()))
    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [152]:
recommend('Spider-Man 3')

[('Spider-Man 2', 0.355139145786159),
 ('Spider-Man', 0.3099927787336151),
 ('Arachnophobia', 0.21401477427459903),
 ('Music and Lyrics', 0.21298151235483997)]

### Debug KNN similarities — inspect tokens and contributions

In [153]:
from sklearn.metrics.pairwise import cosine_similarity

def debug_similarities(title, top_n=10):
    if title not in df['title'].values:
        print('Title not found')
        return
    idx = int(df.index[df['title'] == title][0])
    # get a bunch of neighbors to inspect order
    dists, inds = nn.kneighbors(X[idx], n_neighbors=20)
    dists = dists.flatten()
    inds = inds.flatten()
    sims = 1 - dists
    print('Top candidates (neighbor order):')
    for i, (t, s) in enumerate(zip(df['title'].iloc[inds].tolist(), sims.tolist())):
        print(f"{i+1}. {t}: similarity={s:.4f} (distance={dists[i]:.4f})")

    feature_names = tf.get_feature_names_out()
    q_vec = X[idx].toarray().flatten()
    q_top = [ (feature_names[i], q_vec[i]) for i in q_vec.argsort()[-top_n:][::-1] if q_vec[i]>0 ]
    print('\nTop tokens in query:')
    for name, val in q_top:
        print(f"{name}: {val:.4f}")

    # inspect specific neighbors that concern you
    targets = ['Iron Man 2', 'Arachnophobia', 'Music and Lyrics']
    for target in targets:
        print('\n---')
        if target not in df['title'].values:
            print(f"{target} not in dataset")
            continue
        j = int(df.index[df['title'] == target][0])
        v = X[j].toarray().flatten()
        top_v = [ (feature_names[i], v[i]) for i in v.argsort()[-top_n:][::-1] if v[i]>0 ]
        print(f'Top tokens in {target}:')
        for name, val in top_v:
            print(f"{name}: {val:.4f}")
        # shared token contributions (dot-product style)
        shared = []
        for i in range(len(feature_names)):
            if q_vec[i]>0 and v[i]>0:
                shared.append( (feature_names[i], q_vec[i]*v[i]) )
        shared = sorted(shared, key=lambda x: x[1], reverse=True)[:top_n]
        print(f'\nTop shared tokens with {target}:')
        for name, contrib in shared:
            print(f"{name}: contribution={contrib:.6f}")
        # print similarity explicitly
        sim = cosine_similarity(X[idx], X[j])[0][0]
        print(f'Cosine similarity between query and {target}: {sim:.6f}')

# run debug for Spider-Man 3
debug_similarities('Spider-Man 3')

Top candidates (neighbor order):
1. Spider-Man 3: similarity=1.0000 (distance=0.0000)
2. Spider-Man 2: similarity=0.3551 (distance=0.6449)
3. Spider-Man: similarity=0.3100 (distance=0.6900)
4. Arachnophobia: similarity=0.2140 (distance=0.7860)
5. Music and Lyrics: similarity=0.2130 (distance=0.7870)
6. Iron Man 2: similarity=0.2038 (distance=0.7962)
7. Doubt: similarity=0.1958 (distance=0.8042)
8. The Dark Knight: similarity=0.1905 (distance=0.8095)
9. Logan: similarity=0.1849 (distance=0.8151)
10. Daredevil: similarity=0.1813 (distance=0.8187)
11. Hoffa: similarity=0.1760 (distance=0.8240)
12. Dog Eat Dog: similarity=0.1752 (distance=0.8248)
13. Iron Man: similarity=0.1731 (distance=0.8269)
14. Hillsborough: similarity=0.1721 (distance=0.8279)
15. The Amazing Spider-Man 2: similarity=0.1713 (distance=0.8287)
16. Hairspray: similarity=0.1694 (distance=0.8306)
17. Superman: similarity=0.1685 (distance=0.8315)
18. GoodFellas: similarity=0.1681 (distance=0.8319)
19. Catch Me If You Can: s